# Trascrizione Audio con interfaccia grafica 

1. Cambia opzioni e inizia una nuova sessione: Internet : ON (attivo), Acceleratore : GPU
2. Esegui tutte ( Run All ) 

In [ ]:
# Installazione delle dipendenze
import subprocess
from IPython.display import clear_output

print("Installazione delle dipendenze in corso... (potrebbe volerci un minuto)")
try:
    subprocess.run("apt-get update -qq && apt-get install -y ffmpeg nodejs", shell=True, check=True, capture_output=True, text=True)
    # Rimosso check=True per ignorare i conflitti di dipendenza innocui di Kaggle/Colab (come dask-cuda)
    subprocess.run("pip install -q faster-whisper autocorrect torchaudio yt-dlp", shell=True, capture_output=True, text=True)
    clear_output()
    print("Dipendenze installate con successo.")
except subprocess.CalledProcessError as e:
    clear_output()
    print("Errore durante l'installazione:\n")
    print(e.stdout)
    print(e.stderr)


In [ ]:
# Avvio GUI
import os
import io
import base64
import logging
import warnings
import torchaudio
import yt_dlp
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from faster_whisper import WhisperModel, BatchedInferencePipeline
from autocorrect import Speller
import huggingface_hub

# Soppressione degli avvisi e log non autenticati di Hugging Face
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
warnings.filterwarnings("ignore", message=".*unauthenticated requests.*")
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
huggingface_hub.utils.logging.set_verbosity_error()

# --- 1. STILE MINIMALISTA E LAYOUT ---
clean_layout = widgets.Layout(margin='10px 0px 10px 0px', width='100%')
dropdown_layout = widgets.Layout(width='200px')

# --- 2. DEFINIZIONE DEI WIDGET ---
header = widgets.HTML(value="<h2 style='font-family: sans-serif; color: #333;'>Pipeline di Trascrizione Audio</h2>")

input_type = widgets.ToggleButtons(
    options=['Percorso File Locale', 'Link YouTube'],
    description='Sorgente:',
    button_style='', 
    layout=clean_layout
)

path_input = widgets.Text(
    placeholder='/kaggle/input/...',
    description='Percorso/URL:',
    layout=widgets.Layout(width='80%')
)

model_dropdown = widgets.Dropdown(
    options=['tiny', 'base', 'small', 'medium', 'large-v2', 'large-v3'],
    value='large-v2',
    description='Modello:',
    layout=dropdown_layout
)

lang_dropdown = widgets.Dropdown(
    options=[('Italiano', 'it'), ('Inglese', 'en')],
    value='it',
    description='Lingua:',
    layout=dropdown_layout
)

run_btn = widgets.Button(
    description='Avvia Trascrizione',
    button_style='primary',
    icon='play',
    layout=widgets.Layout(width='250px', height='40px', margin='20px 0px')
)

download_html = widgets.HTML(value="") 
preview_area = widgets.Textarea(
    value='',
    placeholder='L\'anteprima della trascrizione apparirà qui...',
    layout=widgets.Layout(width='100%', height='250px', border='1px solid #e0e0e0')
)
log_output = widgets.Output(layout=widgets.Layout(width='100%', height='150px', overflow='auto', border='1px solid #e0e0e0', padding='5px'))

# --- 3. LOGICA DELLA PIPELINE ---
def process_audio(b):
    run_btn.disabled = True
    preview_area.value = ""
    download_html.value = ""
    
    with log_output:
        clear_output()
        SAMPLING_RATE = 16000
        working_audio = "source_audio.wav"
        
        try:
            print("Avvio del processo...")
            # Step 1: Recupero/Verifica Audio
            if input_type.value == 'Link YouTube':
                print(f"Download dell'audio da YouTube: {path_input.value}...")
                ydl_opts = {
                    'format': 'bestaudio/best',
                    'outtmpl': 'yt_download',
                    'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'wav'}],
                    'quiet': True
                }
                with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                    ydl.download([path_input.value])
                input_path = "yt_download.wav"
            else:
                input_path = path_input.value
                if not os.path.exists(input_path):
                    raise FileNotFoundError(f"File mancante in: {input_path}")
            
            # Step 2: Pre-elaborazione FFmpeg (Rimozione Silenzio e Normalizzazione)
            print("Pre-elaborazione dell'audio in corso...")
            WAV_AUDIO = "step1_trimmed.wav"
            CPS_AUDIO = "step2_norm.wav"
            
            os.system(f'ffmpeg -y -i "{input_path}" -c:a pcm_s16le -ar {SAMPLING_RATE} "{WAV_AUDIO}" -loglevel error')
            os.system(f'ffmpeg -y -i "{WAV_AUDIO}" -af "silenceremove=start_periods=1:start_threshold=-50dB:start_silence=0.3, loudnorm" -c:a pcm_s16le -ar {SAMPLING_RATE} "{CPS_AUDIO}" -loglevel error')
            
            # Step 3: Trascrizione 
            print("Trascrizione in corso...")
            model = WhisperModel(model_dropdown.value, device="cuda", compute_type="float16")
            batched_model = BatchedInferencePipeline(model=model)
            
            segments_gen, _ = batched_model.transcribe(
                CPS_AUDIO, 
                language=lang_dropdown.value,
                beam_size=5, 
                vad_filter=True,
                vad_parameters=dict(min_silence_duration_ms=500),
                batch_size=16,
            )
            
            # Step 4: Post-elaborazione e Correttore Ortografico
            print("Applicazione del controllo ortografico e formattazione...")
            spell = Speller(lang=lang_dropdown.value)
            
            def clean_text(text, lang):
                if lang == 'it':
                    italian_accents = set("àèéìíòóùúÀÈÉÌÍÒÓÙÚ")
                    return ''.join(i for i in text if ord(i) < 128 or i in italian_accents)
                return ''.join(i for i in text if ord(i) < 128)

            final_transcript = []
            for segment in segments_gen:
                clean = clean_text(segment.text, lang_dropdown.value)
                corrected = spell(clean)
                # Nessun timestamp, solo il testo pulito
                line = corrected.strip()
                final_transcript.append(line)
                
            # Unisce tutto con uno spazio per un documento fluido e continuo
            full_text = " ".join(final_transcript)
            preview_area.value = full_text
            
            # Step 5: Generazione Bottone di Download
            b64 = base64.b64encode(full_text.encode()).decode()
            btn_css = "background-color: #2196F3; color: white; padding: 10px 20px; text-decoration: none; border-radius: 4px; font-family: sans-serif; display: inline-block; margin-top: 10px;"
            download_html.value = f'<a href="data:text/plain;charset=utf-8;base64,{b64}" download="trascrizione_{lang_dropdown.value}.txt" style="{btn_css}">Scarica File di Testo</a>'
            print("Fatto!")
            
        except Exception as e:
            print(f"Errore: {str(e)}")
            
        finally:
            run_btn.disabled = False

run_btn.on_click(process_audio)

# --- 4. RENDER GUI ---
ui = widgets.VBox([
    header,
    widgets.HBox([input_type]),
    path_input,
    widgets.HBox([model_dropdown, lang_dropdown], layout=clean_layout),
    run_btn,
    widgets.HTML("<b>Anteprima Trascrizione:</b>"),
    preview_area,
    widgets.HTML("<b>Log della Console:</b>"),
    log_output,
    download_html
])

display(ui)
